<a href="https://colab.research.google.com/github/PatrickSekey/agentic-matching-system/blob/main/Agentic_Profile_Matching.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Set Up the Environment**

In [28]:
# Cell 1: Install required packages
!pip install -q langgraph langchain langchain-openai gradio

In [29]:
# Cell 2: Import all necessary libraries
import os
import json
import re
import getpass
from typing_extensions import TypedDict, List, Dict, Any, Optional
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
import gradio as gr
from datetime import datetime
import random

**Set Up OpenRouter API Key**

In [30]:
# Cell 3: Configure OpenRouter
# You'll be prompted to enter your API key
from google.colab import userdata
try:
    # Try to get from Colab secrets
    OPENROUTER_API_KEY = userdata.get('OPENROUTER_API_KEY')
    print("✅ API key loaded from Colab secrets!")
except:
    # If not in secrets, prompt for it
    OPENROUTER_API_KEY = getpass.getpass("Enter your OpenRouter API Key: ")
    print("✅ API key set!")

os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY

# Initialize LLM with OpenRouter
llm = ChatOpenAI(
    model="openai/gpt-4o-mini",  # You can change to any model on OpenRouter
    temperature=0.3,
    max_tokens=500,
    openai_api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1"
)

print("✅ OpenRouter configured successfully!")
print(f"Model: {llm.model_name}")

Enter your OpenRouter API Key: ··········
✅ API key set!
✅ OpenRouter configured successfully!
Model: openai/gpt-4o-mini


**Define State and Sample Data**

In [31]:
# Cell 4: Define the state structure
class Candidate(TypedDict):
    """Individual candidate profile"""
    id: str
    name: str
    skills: List[str]
    experience: int
    education: str
    summary: str
    match_score: Optional[float]
    strengths: List[str]
    gaps: List[str]
    interview_questions: List[str]
    recommendation: str

class State(TypedDict):
    """Shared state for the matching workflow"""
    conversation_history: List[Dict[str, str]]
    current_query: str
    job_description: str
    required_skills: List[str]
    preferred_skills: List[str]
    min_experience: int
    all_candidates: List[Candidate]
    shortlisted_candidates: List[Candidate]
    top_candidates: List[Candidate]
    final_recommendations: List[Candidate]
    ranking_reasoning: str
    match_report: str
    interview_round: int
    candidate_under_review: Optional[str]
    phase: str

# Cell 5: Create sample candidate data
def create_sample_candidates():
    """Generate mock candidate data for testing"""
    return [
        {
            "id": "C001",
            "name": "Alice Johnson",
            "skills": ["Python", "React", "TypeScript", "Django", "PostgreSQL"],
            "experience": 5,
            "education": "MS in Computer Science, Stanford",
            "summary": "Full-stack developer with 5 years experience",
            "match_score": None,
            "strengths": [],
            "gaps": [],
            "interview_questions": [],
            "recommendation": ""
        },
        {
            "id": "C002",
            "name": "Bob Smith",
            "skills": ["Java", "Spring Boot", "React", "MongoDB", "AWS"],
            "experience": 3,
            "education": "BS in Software Engineering, MIT",
            "summary": "Backend developer with strong cloud skills",
            "match_score": None,
            "strengths": [],
            "gaps": [],
            "interview_questions": [],
            "recommendation": ""
        },
        {
            "id": "C003",
            "name": "Carol White",
            "skills": ["Python", "Django", "Flask", "JavaScript", "React"],
            "experience": 4,
            "education": "PhD in Computer Science, Berkeley",
            "summary": "Full-stack developer with AI/ML experience",
            "match_score": None,
            "strengths": [],
            "gaps": [],
            "interview_questions": [],
            "recommendation": ""
        },
        {
            "id": "C004",
            "name": "David Brown",
            "skills": ["React", "Vue.js", "Node.js", "TypeScript", "GraphQL"],
            "experience": 6,
            "education": "MS in Web Development, USC",
            "summary": "Frontend specialist with UI/UX focus",
            "match_score": None,
            "strengths": [],
            "gaps": [],
            "interview_questions": [],
            "recommendation": ""
        },
        {
            "id": "C005",
            "name": "Eva Martinez",
            "skills": ["Python", "Django", "React", "PostgreSQL", "Docker"],
            "experience": 7,
            "education": "MS in Information Systems, NYU",
            "summary": "Senior full-stack developer and tech lead",
            "match_score": None,
            "strengths": [],
            "gaps": [],
            "interview_questions": [],
            "recommendation": ""
        },
        {
            "id": "C006",
            "name": "Frank Wilson",
            "skills": ["Java", "Spring", "Hibernate", "MySQL"],
            "experience": 2,
            "education": "BS in Computer Engineering, UT Austin",
            "summary": "Backend developer specializing in enterprise apps",
            "match_score": None,
            "strengths": [],
            "gaps": [],
            "interview_questions": [],
            "recommendation": ""
        },
        {
            "id": "C007",
            "name": "Grace Lee",
            "skills": ["Python", "FastAPI", "React", "MongoDB", "AWS"],
            "experience": 4,
            "education": "MS in Data Science, Columbia",
            "summary": "Full-stack developer with data science background",
            "match_score": None,
            "strengths": [],
            "gaps": [],
            "interview_questions": [],
            "recommendation": ""
        },
        {
            "id": "C008",
            "name": "Henry Chen",
            "skills": ["JavaScript", "React", "Angular", "Node.js"],
            "experience": 3,
            "education": "BS in Computer Science, UIUC",
            "summary": "Frontend developer with modern JS frameworks",
            "match_score": None,
            "strengths": [],
            "gaps": [],
            "interview_questions": [],
            "recommendation": ""
        },
        {
            "id": "C009",
            "name": "Irene Kumar",
            "skills": ["Python", "Django", "React", "PostgreSQL", "Redis"],
            "experience": 8,
            "education": "PhD in Computer Science, CMU",
            "summary": "Principal engineer with high-scale systems",
            "match_score": None,
            "strengths": [],
            "gaps": [],
            "interview_questions": [],
            "recommendation": ""
        },
        {
            "id": "C010",
            "name": "Jack Taylor",
            "skills": ["React", "Vue.js", "Svelte", "TypeScript"],
            "experience": 1,
            "education": "BS in Web Design, RISD",
            "summary": "Junior frontend developer with design skills",
            "match_score": None,
            "strengths": [],
            "gaps": [],
            "interview_questions": [],
            "recommendation": ""
        }
    ]

**Define Tools and Helper Functions**

In [32]:
def extract_requirements(jd: str) -> Dict[str, Any]:
    """Parse job description to extract requirements"""
    prompt = f"""
    Analyze this job description and extract requirements:

    {jd}

    Return ONLY a JSON object with these exact fields:
    - required_skills: list of must-have skills
    - preferred_skills: list of nice-to-have skills
    - min_experience: minimum years of experience (integer)

    Example: {{"required_skills": ["Python", "React"], "preferred_skills": ["Django"], "min_experience": 3}}
    """

    try:
        response = llm.invoke([
            SystemMessage(content="You are a skilled HR analyst. Return only valid JSON."),
            HumanMessage(content=prompt)
        ])

        content = response.content
        json_match = re.search(r'\{.*\}', content, re.DOTALL)
        if json_match:
            data = json.loads(json_match.group())
            return {
                "required_skills": data.get("required_skills", ["Python", "React"]),
                "preferred_skills": data.get("preferred_skills", ["Django", "PostgreSQL"]),
                "min_experience": data.get("min_experience", 3)
            }
    except Exception as e:
        print(f"Error parsing requirements: {e}")

    return {"required_skills": ["Python", "React"], "preferred_skills": ["Django", "PostgreSQL"], "min_experience": 3}

def compare_candidates(candidate_ids: List[str], all_candidates: List[Candidate]) -> Dict[str, str]:
    """
    Head-to-head comparison of candidates
    COMPLETE FIX: Handles None values properly
    """
    candidates = [c for c in all_candidates if c["id"] in candidate_ids]

    comparison_text = "## 📊 Candidate Comparison\n\n"
    for i, candidate in enumerate(candidates, 1):
        # FIX 1: Handle None match_score
        match_score = candidate.get('match_score')
        if match_score is None:
            match_score = 0.0

        # FIX 2: Handle None or empty strengths
        strengths = candidate.get('strengths')
        if strengths is None or len(strengths) == 0:
            strengths = ['None specified']

        # FIX 3: Handle None or empty gaps
        gaps = candidate.get('gaps')
        if gaps is None or len(gaps) == 0:
            gaps = ['None']

        comparison_text += f"### {i}. {candidate['name']}\n"
        comparison_text += f"- **Skills**: {', '.join(candidate['skills'])}\n"
        comparison_text += f"- **Experience**: {candidate['experience']} years\n"
        comparison_text += f"- **Match Score**: {match_score:.2f}\n"
        comparison_text += f"- **Strengths**: {', '.join(strengths)}\n"
        comparison_text += f"- **Gaps**: {', '.join(gaps)}\n\n"

    return {"comparison": comparison_text}

def generate_interview_questions(candidate_id: str, all_candidates: List[Candidate]) -> List[str]:
    """Generate interview questions for a candidate"""
    candidate = next((c for c in all_candidates if c["id"] == candidate_id), None)
    if not candidate:
        return ["No candidate found"]

    prompt = f"""
    Generate 5 interview questions for this candidate:

    Name: {candidate['name']}
    Skills: {', '.join(candidate['skills'])}
    Experience: {candidate['experience']} years
    Summary: {candidate['summary']}

    Focus on technical skills. Return just the questions, one per line.
    """

    try:
        response = llm.invoke([
            SystemMessage(content="Return 5 interview questions, one per line."),
            HumanMessage(content=prompt)
        ])
        questions = [q.strip() for q in response.content.split("\n") if q.strip()]
        return questions[:5]
    except:
        return ["Tell me about your experience with the required technologies."]

print("✅ Helper functions loaded successfully!")
print("   - extract_requirements: Extracts JD requirements")
print("   - compare_candidates: Compares candidates (FIXED for None values)")
print("   - generate_interview_questions: Creates interview questions")

✅ Helper functions loaded successfully!
   - extract_requirements: Extracts JD requirements
   - compare_candidates: Compares candidates (FIXED for None values)
   - generate_interview_questions: Creates interview questions


**Define LangGraph Nodes**

In [33]:
def parse_jd_node(state: State) -> Dict[str, Any]:
    """Parse job description and extract requirements"""
    if not state.get("job_description"):
        return {"phase": "error", "conversation_history": [{"role": "system", "content": "No job description provided"}]}

    requirements = extract_requirements(state["job_description"])

    return {
        "required_skills": requirements["required_skills"],
        "preferred_skills": requirements["preferred_skills"],
        "min_experience": requirements["min_experience"],
        "phase": "requirements_extracted",
        "conversation_history": state.get("conversation_history", []) + [
            {"role": "system", "content": f"📋 Extracted requirements: Required: {requirements['required_skills']}"}
        ]
    }

def search_resumes_node(state: State) -> Dict[str, Any]:
    """Search and filter candidates based on requirements"""
    candidates = state.get("all_candidates", [])
    required_skills = state.get("required_skills", [])
    min_experience = state.get("min_experience", 0)

    filtered_candidates = []
    for candidate in candidates:
        # Check minimum experience
        if candidate["experience"] < min_experience:
            continue

        # Check skill match (at least 50%)
        match_count = sum(1 for req in required_skills
                         if any(req.lower() in s.lower() or s.lower() in req.lower()
                               for s in candidate["skills"]))

        if len(required_skills) > 0 and match_count / len(required_skills) >= 0.5:
            filtered_candidates.append(candidate)

    filtered_candidates.sort(key=lambda x: x["experience"], reverse=True)

    return {
        "shortlisted_candidates": filtered_candidates[:20],
        "phase": "candidates_searched",
        "conversation_history": state.get("conversation_history", []) + [
            {"role": "system", "content": f"🔍 Found {len(filtered_candidates[:20])} matching candidates"}
        ]
    }

def rank_candidates_node(state: State) -> Dict[str, Any]:
    """Rank candidates based on job requirements"""
    candidates = state.get("shortlisted_candidates", [])
    required_skills = state.get("required_skills", [])
    preferred_skills = state.get("preferred_skills", [])

    scored_candidates = []

    for candidate in candidates:
        # Calculate matches
        required_match = sum(1 for req in required_skills
                           if any(req.lower() in s.lower() or s.lower() in req.lower()
                                 for s in candidate["skills"]))
        preferred_match = sum(1 for pref in preferred_skills
                            if any(pref.lower() in s.lower() or s.lower() in pref.lower()
                                  for s in candidate["skills"]))

        # Calculate scores
        required_score = required_match / len(required_skills) if required_skills else 0
        preferred_score = preferred_match / len(preferred_skills) if preferred_skills else 0
        exp_score = min(candidate["experience"] / 10, 1.0)

        # Weighted total
        total_score = (required_score * 0.5) + (preferred_score * 0.3) + (exp_score * 0.2)

        # Determine strengths and gaps
        strengths = [skill for skill in candidate["skills"]
                    if any(req.lower() in skill.lower() or skill.lower() in req.lower()
                          for req in required_skills)]
        gaps = [req for req in required_skills
               if not any(req.lower() in skill.lower() or skill.lower() in req.lower()
                         for skill in candidate["skills"])]

        # Create candidate copy with scores
        candidate_copy = candidate.copy()
        candidate_copy["match_score"] = total_score
        candidate_copy["strengths"] = strengths[:3]
        candidate_copy["gaps"] = gaps[:3]
        candidate_copy["recommendation"] = "hire" if total_score > 0.8 else "consider" if total_score > 0.5 else "reject"

        scored_candidates.append(candidate_copy)

    scored_candidates.sort(key=lambda x: x["match_score"], reverse=True)
    top_10 = scored_candidates[:10]

    # Safely get top candidate name and score
    top_name = "None"
    top_score = 0.0
    if top_10 and len(top_10) > 0:
        top_name = top_10[0].get("name", "Unknown")
        top_score = top_10[0].get("match_score", 0.0)

    return {
        "top_candidates": top_10,
        "phase": "candidates_ranked",
        "ranking_reasoning": f"📈 Ranked {len(scored_candidates)} candidates based on weighted scoring",
        "conversation_history": state.get("conversation_history", []) + [
            {"role": "system", "content": f"🏆 Top match: {top_name} - {top_score:.2f}"}
        ]
    }

def generate_report_node(state: State) -> Dict[str, Any]:
    """Generate detailed match report"""
    top_candidates = state.get("top_candidates", [])

    report = "# 📊 Detailed Match Report\n\n"

    if not top_candidates:
        report += "No candidates found matching the requirements.\n"
    else:
        for i, candidate in enumerate(top_candidates[:10], 1):
            # Safely get values with defaults
            name = candidate.get("name", "Unknown")
            experience = candidate.get("experience", 0)
            match_score = candidate.get("match_score", 0.0)
            if match_score is None:
                match_score = 0.0
            recommendation = candidate.get("recommendation", "N/A")
            strengths = candidate.get("strengths", ["None"])
            if not strengths:
                strengths = ["None"]
            gaps = candidate.get("gaps", ["None"])
            if not gaps:
                gaps = ["None"]

            report += f"## {i}. {name}\n"
            report += f"- **Experience**: {experience} years\n"
            report += f"- **Match Score**: {match_score:.2f}\n"
            report += f"- **Recommendation**: {recommendation}\n"
            report += f"- **Strengths**: {', '.join(strengths)}\n"
            report += f"- **Gaps**: {', '.join(gaps)}\n\n"

    return {
        "match_report": report,
        "phase": "report_generated"
    }

def human_feedback_node(state: State) -> Dict[str, Any]:
    """
    Process human feedback and adjust recommendations
    FIXED: Added safety checks for all query types
    """
    current_query = state.get("current_query", "").lower()

    if not current_query:
        return {"phase": "feedback_processed"}

    # Handle COMPARE query
    if "compare" in current_query:
        # Safety check: Need at least 3 candidates
        top_candidates = state.get("top_candidates", [])
        if len(top_candidates) < 3:
            return {
                "phase": "feedback_processed",
                "match_report": state.get("match_report", "") + "\n\n⚠️ Not enough candidates to compare (need 3).",
                "conversation_history": state.get("conversation_history", []) + [
                    {"role": "user", "content": current_query},
                    {"role": "system", "content": "⚠️ Need at least 3 candidates for comparison"}
                ]
            }

        # Get valid candidate IDs
        top_3_ids = []
        for c in top_candidates[:3]:
            if c.get("id"):
                top_3_ids.append(c["id"])

        if len(top_3_ids) < 3:
            return {
                "phase": "feedback_processed",
                "match_report": state.get("match_report", "") + "\n\n⚠️ Invalid candidate IDs found.",
                "conversation_history": state.get("conversation_history", []) + [
                    {"role": "user", "content": current_query},
                    {"role": "system", "content": "⚠️ Invalid candidate IDs"}
                ]
            }

        # Generate comparison
        comparison = compare_candidates(top_3_ids, state.get("all_candidates", []))
        return {
            "match_report": state.get("match_report", "") + "\n\n" + comparison["comparison"],
            "phase": "feedback_processed",
            "conversation_history": state.get("conversation_history", []) + [
                {"role": "user", "content": current_query},
                {"role": "system", "content": "📊 Generated comparison of top 3 candidates"}
            ]
        }

    # Handle EXPLAIN/WHY query
    elif "why" in current_query or "reason" in current_query or "explain" in current_query:
        reasoning = state.get("ranking_reasoning", "Candidates ranked based on weighted scoring (Required: 50%, Preferred: 30%, Experience: 20%).")
        return {
            "match_report": state.get("match_report", "") + "\n\n## 🤔 Ranking Explanation\n" + reasoning,
            "phase": "feedback_processed",
            "conversation_history": state.get("conversation_history", []) + [
                {"role": "user", "content": current_query},
                {"role": "system", "content": "📝 Ranking explanation provided"}
            ]
        }

    # Handle INTERVIEW query
    elif "interview" in current_query or "questions" in current_query:
        top_candidates = state.get("top_candidates", [])
        if not top_candidates:
            return {
                "phase": "feedback_processed",
                "match_report": state.get("match_report", "") + "\n\n⚠️ No candidates found for interview.",
                "conversation_history": state.get("conversation_history", []) + [
                    {"role": "user", "content": current_query},
                    {"role": "system", "content": "⚠️ No candidates available"}
                ]
            }

        top_candidate = top_candidates[0]
        if top_candidate and top_candidate.get("id"):
            questions = generate_interview_questions(top_candidate["id"], state.get("all_candidates", []))
            q_text = "\n".join([f"{i+1}. {q}" for i, q in enumerate(questions)])
            return {
                "match_report": state.get("match_report", "") + "\n\n## 🎯 Interview Questions\n" + q_text,
                "phase": "feedback_processed",
                "conversation_history": state.get("conversation_history", []) + [
                    {"role": "user", "content": current_query},
                    {"role": "system", "content": f"🎤 Generated interview questions for {top_candidate['name']}"}
                ]
            }
        else:
            return {
                "phase": "feedback_processed",
                "match_report": state.get("match_report", "") + "\n\n⚠️ Invalid candidate for interview.",
                "conversation_history": state.get("conversation_history", []) + [
                    {"role": "user", "content": current_query},
                    {"role": "system", "content": "⚠️ Invalid candidate"}
                ]
            }

    # Handle PROFILE/STRENGTHS query
    elif "strength" in current_query or "gap" in current_query or "profile" in current_query:
        top_candidates = state.get("top_candidates", [])
        if not top_candidates:
            return {
                "phase": "feedback_processed",
                "match_report": state.get("match_report", "") + "\n\n⚠️ No candidates found.",
                "conversation_history": state.get("conversation_history", []) + [
                    {"role": "user", "content": current_query},
                    {"role": "system", "content": "⚠️ No candidates available"}
                ]
            }

        top_candidate = top_candidates[0]
        strengths = top_candidate.get("strengths", ["None"])
        if not strengths:
            strengths = ["None"]
        gaps = top_candidate.get("gaps", ["None"])
        if not gaps:
            gaps = ["None"]

        profile_text = f"""
## 👤 Candidate Profile: {top_candidate.get('name', 'Unknown')}

### ✅ Strengths
- {', '.join(strengths)}

### ❌ Gaps
- {', '.join(gaps)}

### 📊 Match Score: {top_candidate.get('match_score', 0.0):.2f}
### 📝 Recommendation: {top_candidate.get('recommendation', 'N/A')}
"""
        return {
            "match_report": state.get("match_report", "") + "\n\n" + profile_text,
            "phase": "feedback_processed",
            "conversation_history": state.get("conversation_history", []) + [
                {"role": "user", "content": current_query},
                {"role": "system", "content": f"📋 Profile analysis for {top_candidate.get('name', 'Unknown')}"}
            ]
        }

    # Default: Log the query
    return {
        "phase": "feedback_processed",
        "conversation_history": state.get("conversation_history", []) + [
            {"role": "user", "content": current_query},
            {"role": "system", "content": "✅ Feedback processed"}
        ]
    }

def final_recommendation_node(state: State) -> Dict[str, Any]:
    """Generate final hire/no-hire recommendations"""
    top_candidates = state.get("top_candidates", [])

    recommendations = []
    for candidate in top_candidates[:5]:
        match_score = candidate.get("match_score", 0.0)
        if match_score is None:
            match_score = 0.0

        if match_score >= 0.8:
            recommendation = "✅ HIRE"
        elif match_score >= 0.6:
            recommendation = "🔄 CONSIDER"
        else:
            recommendation = "❌ REJECT"

        strengths = candidate.get("strengths", ["None"])
        if not strengths:
            strengths = ["None"]
        gaps = candidate.get("gaps", ["None"])
        if not gaps:
            gaps = ["None"]

        recommendations.append({
            "name": candidate.get("name", "Unknown"),
            "score": match_score,
            "recommendation": recommendation,
            "strengths": strengths,
            "gaps": gaps
        })

    final_report = "# 🎯 Final Recommendations\n\n"
    for rec in recommendations:
        final_report += f"## {rec['name']}\n"
        final_report += f"- **Score**: {rec['score']:.2f}\n"
        final_report += f"- **Decision**: {rec['recommendation']}\n"
        final_report += f"- **Strengths**: {', '.join(rec['strengths'])}\n"
        final_report += f"- **Gaps**: {', '.join(rec['gaps'])}\n\n"

    return {
        "final_recommendations": recommendations,
        "match_report": state.get("match_report", "") + "\n\n" + final_report,
        "phase": "final"
    }

print("✅ All workflow nodes loaded successfully!")
print("   - parse_jd_node: Parses job description")
print("   - search_resumes_node: Filters candidates")
print("   - rank_candidates_node: Ranks candidates")
print("   - generate_report_node: Creates match report")
print("   - human_feedback_node:")
print("   - final_recommendation_node: Generates final decisions")

✅ All workflow nodes loaded successfully!
   - parse_jd_node: Parses job description
   - search_resumes_node: Filters candidates
   - rank_candidates_node: Ranks candidates
   - generate_report_node: Creates match report
   - human_feedback_node:
   - final_recommendation_node: Generates final decisions


**Build and Run the Workflow**

In [34]:
# Cell 8: Build the workflow
def build_matching_workflow():
    workflow = StateGraph(State)

    workflow.add_node("parse_jd", parse_jd_node)
    workflow.add_node("search_resumes", search_resumes_node)
    workflow.add_node("rank_candidates", rank_candidates_node)
    workflow.add_node("generate_report", generate_report_node)
    workflow.add_node("human_feedback", human_feedback_node)
    workflow.add_node("final_recommendation", final_recommendation_node)

    workflow.add_edge(START, "parse_jd")
    workflow.add_edge("parse_jd", "search_resumes")
    workflow.add_edge("search_resumes", "rank_candidates")
    workflow.add_edge("rank_candidates", "generate_report")
    workflow.add_edge("generate_report", "human_feedback")
    workflow.add_edge("human_feedback", "final_recommendation")
    workflow.add_edge("final_recommendation", END)

    return workflow.compile()

# Cell 9: Test the workflow
def test_matching():
    """Test the matching agent with default JD"""
    jd = """Senior Full-Stack Developer. We are looking for an experienced developer
    with Python, React, Django, and PostgreSQL skills. Minimum 3 years of experience."""

    state: State = {
        "conversation_history": [],
        "current_query": "",
        "job_description": jd,
        "required_skills": [],
        "preferred_skills": [],
        "min_experience": 0,
        "all_candidates": create_sample_candidates(),
        "shortlisted_candidates": [],
        "top_candidates": [],
        "final_recommendations": [],
        "ranking_reasoning": "",
        "match_report": "",
        "interview_round": 1,
        "candidate_under_review": None,
        "phase": "start"
    }

    app = build_matching_workflow()
    result = app.invoke(state)

    print("✅ MATCHING COMPLETE!")
    print("=" * 60)
    print(result.get("match_report", "No report generated"))
    return result

# Run the test
result = test_matching()

✅ MATCHING COMPLETE!
# 📊 Detailed Match Report

## 1. Irene Kumar
- **Experience**: 8 years
- **Match Score**: 0.66
- **Recommendation**: consider
- **Strengths**: Python, Django, React
- **Gaps**: None

## 2. Eva Martinez
- **Experience**: 7 years
- **Match Score**: 0.64
- **Recommendation**: consider
- **Strengths**: Python, Django, React
- **Gaps**: None

## 3. Alice Johnson
- **Experience**: 5 years
- **Match Score**: 0.60
- **Recommendation**: consider
- **Strengths**: Python, React, Django
- **Gaps**: None

## 4. Carol White
- **Experience**: 4 years
- **Match Score**: 0.46
- **Recommendation**: reject
- **Strengths**: Python, Django, React
- **Gaps**: PostgreSQL

## 5. Grace Lee
- **Experience**: 4 years
- **Match Score**: 0.33
- **Recommendation**: reject
- **Strengths**: Python, React
- **Gaps**: Django, PostgreSQL



# 🎯 Final Recommendations

## Irene Kumar
- **Score**: 0.66
- **Decision**: 🔄 CONSIDER
- **Strengths**: Python, Django, React
- **Gaps**: None

## Eva Martinez
-

**Launch Gradio Interface**

In [35]:
# Cell 10: Launch Gradio Interface
def run_matching_agent(job_description: str, query: str = ""):
    """Main function to run the matching agent with Gradio"""
    state: State = {
        "conversation_history": [],
        "current_query": query,
        "job_description": job_description,
        "required_skills": [],
        "preferred_skills": [],
        "min_experience": 0,
        "all_candidates": create_sample_candidates(),
        "shortlisted_candidates": [],
        "top_candidates": [],
        "final_recommendations": [],
        "ranking_reasoning": "",
        "match_report": "",
        "interview_round": 1,
        "candidate_under_review": None,
        "phase": "start"
    }

    app = build_matching_workflow()
    result = app.invoke(state)

    match_report = result.get("match_report", "No report generated")
    top_candidates = result.get("top_candidates", [])

    candidate_list = "## 🏆 Top Candidates\n\n"
    if top_candidates:
        for i, candidate in enumerate(top_candidates[:10], 1):
            candidate_list += f"### {i}. {candidate['name']}\n"
            candidate_list += f"- Match Score: {candidate.get('match_score', 0):.2f}\n"
            candidate_list += f"- Recommendation: {candidate.get('recommendation', 'N/A')}\n"
            candidate_list += f"- Skills: {', '.join(candidate['skills'][:5])}\n"
            candidate_list += f"- Experience: {candidate['experience']} years\n\n"
    else:
        candidate_list += "No candidates found.\n"

    conversation = result.get("conversation_history", [])
    conv_text = "## 💬 Conversation Log\n\n"
    for msg in conversation:
        role = msg.get("role", "unknown")
        content = msg.get("content", "")
        conv_text += f"**{role.upper()}**: {content}\n\n"

    status_text = f"### Status: {result.get('phase', 'start').upper().replace('_', ' ')}"

    return match_report, candidate_list, conv_text, status_text

# Launch the Gradio interface
with gr.Blocks(title="Agentic Profile Matching", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🤖 Agentic Profile Matching System")
    gr.Markdown("Multi-round screening with LangGraph agents")

    with gr.Row():
        jd_input = gr.Textbox(
            label="Job Description",
            value="Senior Full-Stack Developer. Need Python, React, Django, PostgreSQL. Min 3 years experience.",
            lines=5
        )
        query_input = gr.Textbox(
            label="Query (Optional)",
            placeholder="Compare candidates, Explain rankings, Generate interview questions...",
            lines=2
        )

    run_button = gr.Button("🚀 Run Matching", variant="primary")
    status_output = gr.Markdown("### Status: Ready")

    with gr.Tabs():
        with gr.TabItem("📊 Match Report"):
            report_output = gr.Markdown()
        with gr.TabItem("🏆 Top Candidates"):
            candidates_output = gr.Markdown()
        with gr.TabItem("💬 Conversation"):
            conv_output = gr.Markdown()

    run_button.click(
        fn=run_matching_agent,
        inputs=[jd_input, query_input],
        outputs=[report_output, candidates_output, conv_output, status_output]
    )

    gr.Markdown("### 💡 Example Queries")
    for q in ["Compare the top 3 candidates", "Why did Alice rank higher?", "Generate interview questions"]:
        gr.Markdown(f"- `{q}`")

# Launch the app - this will give you a public URL
print("🚀 Launching Gradio interface...")
demo.launch(share=True)

/tmp/ipykernel_1702/2306772074.py:51: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(title="Agentic Profile Matching", theme=gr.themes.Soft()) as demo:


🚀 Launching Gradio interface...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5c7fd1cf19fa316056.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


**Test Cases**

In [36]:
# ============================================================
# FINAL CELL: Comprehensive Test Suite
# Run this LAST after everything is set up
# ============================================================

def run_comprehensive_tests():
    """Run all test scenarios with detailed output"""

    print("\n" + "="*70)
    print("🚀 STARTING COMPREHENSIVE TEST SUITE")
    print("Testing Agentic Profile Matching System")
    print("="*70)

    jd = """Senior Full-Stack Developer. We are looking for an experienced developer
    with Python, React, Django, and PostgreSQL skills. Minimum 3 years of experience.
    Nice to have: AWS, Docker, and GraphQL experience."""

    test_cases = [
        {
            "name": "TEST 1: Basic Matching",
            "query": "",
            "description": "Tests core workflow execution and ranking"
        },
        {
            "name": "TEST 2: Candidate Comparison",
            "query": "Compare the top 3 candidates",
            "description": "Tests comparative analysis tool"
        },
        {
            "name": "TEST 3: Ranking Explanation",
            "query": "Why did Alice rank higher than Bob?",
            "description": "Tests transparency and reasoning capabilities"
        },
        {
            "name": "TEST 4: Interview Questions",
            "query": "Generate interview questions for the top candidate",
            "description": "Tests interview question generation"
        },
        {
            "name": "TEST 5: Strengths & Gaps",
            "query": "Show me the strengths and gaps of the top candidate",
            "description": "Tests detailed candidate profiling"
        },
        {
            "name": "TEST 6: Multiple Interactions",
            "query": "Compare the top 3 candidates. Also, why did Alice rank higher?",
            "description": "Tests handling of complex queries"
        },
        {
            "name": "TEST 7: Edge Case - Non-standard JD",
            "query": "",
            "jd": """We need a JavaScript developer with React experience.
            Should know modern frameworks. 2+ years preferred.""",
            "description": "Tests system with different job description"
        }
    ]

    test_results = []

    for i, test in enumerate(test_cases, 1):
        print(f"\n{'='*70}")
        print(f"📝 {test['name']}")
        print(f"📖 Description: {test['description']}")
        print(f"🔍 Query: '{test.get('query', 'No query')}'")
        print('='*70)

        try:
            # Run the test
            report, candidates, conv, status = run_matching_agent(
                test.get('jd', jd),
                test.get('query', '')
            )

            # Analyze results
            print(f"\n✅ Status: {status}")
            print(f"📊 Report length: {len(report)} characters")
            print(f"👥 Candidate entries: {candidates.count('###')}")
            print(f"💬 Conversation entries: {conv.count('**')}")

            # Show report snippet
            lines = report.split('\n')
            print(f"\n📋 Report Preview:")
            print('-' * 50)
            for line in lines[:15]:  # Show first 15 lines
                if line.strip():
                    print(line[:100])  # Truncate long lines
            if len(lines) > 15:
                print(f"... and {len(lines)-15} more lines")
            print('-' * 50)

            test_results.append({
                "name": test['name'],
                "status": "✅ PASSED",
                "report_length": len(report),
                "candidates": candidates.count('###')
            })

        except Exception as e:
            print(f"\n❌ ERROR: {str(e)}")
            test_results.append({
                "name": test['name'],
                "status": "❌ FAILED",
                "error": str(e)
            })

    # Print Summary
    print("\n" + "="*70)
    print("📊 TEST SUMMARY")
    print("="*70)

    passed = sum(1 for r in test_results if "PASSED" in r['status'])
    total = len(test_results)

    print(f"\n✅ Passed: {passed}/{total}")
    print(f"❌ Failed: {total-passed}/{total}")

    print("\n📋 Detailed Results:")
    for result in test_results:
        status_icon = "✅" if "PASSED" in result['status'] else "❌"
        print(f"  {status_icon} {result['name']}: {result['status']}")
        if 'report_length' in result:
            print(f"     - Report: {result['report_length']} characters")
            print(f"     - Candidates: {result['candidates']}")

    return test_results

# ============================================================
# RUN THE TESTS
# ============================================================
print("\n🚀 Running all test scenarios...")
results = run_comprehensive_tests()

# ============================================================
# OPTIONAL: Launch Gradio after testing
# ============================================================
print("\n" + "="*70)
print("💡 Options:")
print("1. Tests completed! You can now launch the Gradio interface")
print("2. To launch Gradio, uncomment the line below:")
print("   demo.launch(share=True)")
print("="*70)


demo.launch(share=True)


🚀 Running all test scenarios...

🚀 STARTING COMPREHENSIVE TEST SUITE
Testing Agentic Profile Matching System

📝 TEST 1: Basic Matching
📖 Description: Tests core workflow execution and ranking
🔍 Query: ''

✅ Status: ### Status: FINAL
📊 Report length: 1439 characters
👥 Candidate entries: 5
💬 Conversation entries: 6

📋 Report Preview:
--------------------------------------------------
# 📊 Detailed Match Report
## 1. Eva Martinez
- **Experience**: 7 years
- **Match Score**: 0.74
- **Recommendation**: consider
- **Strengths**: Python, Django, React
- **Gaps**: None
## 2. Irene Kumar
- **Experience**: 8 years
- **Match Score**: 0.66
- **Recommendation**: consider
- **Strengths**: Python, Django, React
- **Gaps**: None
... and 57 more lines
--------------------------------------------------

📝 TEST 2: Candidate Comparison
📖 Description: Tests comparative analysis tool
🔍 Query: 'Compare the top 3 candidates'

✅ Status: ### Status: FINAL
📊 Report length: 1999 characters
👥 Candidate entries: 5
